# 📓 Docling Hierarchical 512 Ingestion

**Autor:** Sakina Ahmadi
**Beschreibung:** Dieses Notebook indexiert 3.000 Docling-geparste Papers
in einer neuen Qdrant-Collection (`docling_hierarchical_512`) mit INT8-Quantisierung.

## Ziel
1. **Collection erstellen** – mit HNSW-Index, Cosine Similarity und INT8
2. **3.000 Docling-Papers laden** – aus dem Kaggle-Dataset
3. **Hybrides Chunking** – Markdown-Überschriften + rekursiver Fallback (512 Tokens)
4. **Embedding mit BGE-M3** – 1024 Dimensionen auf GPU
5. **Batch-Upload zu Qdrant** – 32 Chunks pro Batch
6. **Checkpoint-System** – Fortschritt wird nach jedem Dokument gespeichert

---
## 1. Installation & Importe

Wir installieren die benötigten Pakete und importieren alle Bibliotheken.

In [1]:
# =====================================================================
# 1. DEPENDENCIES & IMPORTS
# =====================================================================
!pip install -q langchain-text-splitters langchain-huggingface sentence-transformers qdrant-client tqdm

import os
import re
import json
import time
import hashlib
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import qdrant_client
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from kaggle_secrets import UserSecretsClient

---
## 2. Konfiguration

Hier definieren wir alle Parameter für die Ingestion:
- **Parser:** Docling
- **Chunking:** Hierarchisch (Markdown-Überschriften + rekursiv)
- **Chunk-Größe:** 512 Tokens (mit 10% Overlap)
- **Embedding:** BAAI/bge-m3 (1024 Dimensionen)
- **Collection:** `docling_hierarchical_512`
- **Quantisierung:** INT8 (Scalar Quantization)

In [1]:
# =====================================================================
# 2. DYNAMISCHE MATRIX-KONFIGURATION
# =====================================================================
MD_DIR = "/kaggle/input/datasets/sakinaahmadi/extracted-markdown-docling/markdown"
EVAL_SET_PATH = "/kaggle/input/datasets/sakinaahmadi/automl-ground-truth-100/automl_ground_truth_100.json"

PARSER_NAME = "docling"
STRATEGY_NAME = "hierarchical"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 51  # Exakt 10% Überlappung

COLLECTION_NAME = f"{PARSER_NAME}_{STRATEGY_NAME}_{CHUNK_SIZE}"
CHECKPOINT_FILE = f"/kaggle/working/checkpoint_{COLLECTION_NAME}.txt"

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda"

---
## 3. Hybrid Chunker Klasse

Die `ArxivHybridChunker`-Klasse implementiert:
1. **Markdown-Header-Split** – Trennung nach `#`, `##`, `###` Überschriften
2. **Rekursiver Fallback** – Falls ein Abschnitt länger als 512 Tokens ist
3. **Metadaten-Fusion** – Titel, Abstract und Section werden in den Chunk-Text eingefügt

In [1]:
# =====================================================================
# 3. HYBRID ARXIV CHUNKER KLASSE
# =====================================================================
class ArxivHybridChunker:
    def __init__(self, chunk_size=512, chunk_overlap=51):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        self.headers_to_split_on = [
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ]
        self.markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=self.headers_to_split_on,
            strip_headers=False
        )
        self.fallback_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def chunk_and_fuse(self, md_content, paper_id, meta_lookup):
        paper_meta = meta_lookup.get(paper_id, {"title": "Unknown", "abstract": ""})
        paper_title = paper_meta.get("title", "Unknown")
        paper_abstract = paper_meta.get("abstract", "")[:250]
        
        header_splits = self.markdown_splitter.split_text(md_content)
        final_fused_chunks = []
        chunk_idx = 0
        
        for section in header_splits:
            current_section = section.metadata.get("Header_2", section.metadata.get("Header_1", "Main Body"))
            sub_chunks = self.fallback_splitter.split_text(section.page_content)
            
            for sub_text in sub_chunks:
                fused_text = (
                    f"TITLE: {paper_title}\n"
                    f"ABSTRACT: {paper_abstract}...\n"
                    f"SECTION: {current_section}\n\n"
                    f"CHUNK CONTENT:\n{sub_text}"
                )
                
                final_fused_chunks.append({
                    "paper_id": paper_id,
                    "chunk_index": chunk_idx,
                    "text": fused_text,
                    "metadata": {
                        "title": paper_title,
                        "section": current_section,
                        "chunk_size": self.chunk_size,
                        "chunk_method": "hybrid_markdown"
                    }
                })
                chunk_idx += 1
                
        return final_fused_chunks

---
## 4. Textbereinigung

Die `clean_scientific_markdown()`-Funktion bereinigt den extrahierten Text:
- Entfernt doppelte Leerzeilen
- Behebt Zeilenumbrüche bei Wörtern (z.B. "neu-\nmerk" → "neu merk")

In [1]:
# =====================================================================
# 4. TEXTBEREINIGUNG
# =====================================================================
def clean_scientific_markdown(raw_text):
    cleaned = re.sub(r'\n{3,}', '\n\n', raw_text)
    cleaned = re.sub(r'(?<=[a-zA-Z])-\n(?=[a-zA-Z])', '', cleaned)
    return cleaned

---
## 5. Hauptfunktion

Die Hauptfunktion führt folgende Schritte aus:
1. **Embedding-Modell laden** – BGE-M3 auf GPU
2. **Qdrant-Collection erstellen** – mit INT8-Quantisierung und On-Disk HNSW
3. **Gold-Standard Metadaten laden** – für Titel und Abstract
4. **Checkpoint-System** – Fortschritt wird nach jedem Dokument gespeichert
5. **Ingestion-Schleife** – 3.000 Papers chunken, embedden und hochladen

In [1]:
# =====================================================================
# 5. EXECUTION PIPELINE
# =====================================================================
def main():
    print(f"🖥️ Initialisiere SentenceTransformer ({EMBEDDING_MODEL_NAME}) auf GPU...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
    
    print("🌐 Verbinde über Kaggle Secrets mit Qdrant...")
    user_secrets = UserSecretsClient()
    client = qdrant_client.QdrantClient(
        url=user_secrets.get_secret("QDRANT_URL"), 
        api_key=user_secrets.get_secret("QDRANT_API_KEY"),
        timeout=60.0
    )

    # Erzeuge frische Collection mit On-Disk HNSW und INT8-Kompression
    if not client.collection_exists(COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
            hnsw_config={"on_disk": True},
            quantization_config={
                "scalar": {
                    "type": "int8",
                    "always_ram": False
                }
            }
        )
        print(f"✅ Neue Collection '{COLLECTION_NAME}' (INT8, On-Disk) angelegt.")

    # Gold-Standard Metadaten-Lookup laden
    meta_lookup = {}
    if os.path.exists(EVAL_SET_PATH):
        print(f"📖 Lade Metadaten für das Fusing aus: {EVAL_SET_PATH}")
        with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
            try:
                gold_data = json.load(f)
                for item in gold_data:
                    if "arxiv_id" in item:
                        meta_lookup[item["arxiv_id"]] = {
                            "title": item.get("title", "Unknown"),
                            "abstract": item.get("expected_context", "")[:300]
                        }
            except json.JSONDecodeError:
                f.seek(0)
                for line in f:
                    if line.strip() and line.strip() not in ["[", "]", "],"]:
                        try:
                            if line.strip().endswith(","):
                                line = line.strip()[:-1]
                            data = json.loads(line)
                            if "arxiv_id" in data:
                                meta_lookup[data["arxiv_id"]] = {
                                    "title": data.get("title", "Unknown"),
                                    "abstract": data.get("expected_context", "")[:300]
                                }
                        except:
                            continue

    # Checkpoint auslesen
    done_files = []
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            done_files = f.read().splitlines()

    if not os.path.exists(MD_DIR):
        print(f"❌ FEHLER: Quellpfad nicht gefunden: {MD_DIR}")
        return

    # KORREKTUR: Alle Dateien laden, sortieren und strikt auf die ersten 3000 Papers begrenzen!
    all_files = [f for f in os.listdir(MD_DIR) if f.endswith(".md")]
    all_files.sort()
    all_files = all_files[:3000]
    
    files_to_process = [f for f in all_files if f not in done_files]
    print(f"⏮️ Bereits indiziert von den 3000 Ziel-Arbeiten: {len(done_files)}")
    print(f"🔥 Starte Pipeline für die verbleibenden {len(files_to_process)} Dokumente...")
    
    chunker = ArxivHybridChunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    UPLOAD_BATCH_SIZE = 32

    for filename in tqdm(files_to_process, desc="Ingestion Progress"):
        file_path = os.path.join(MD_DIR, filename)
        paper_id = filename.replace(".md", "").replace("_", ".").strip()
        
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                raw_md = f.read()
            
            cleaned_md = clean_scientific_markdown(raw_md)
            processed_chunks = chunker.chunk_and_fuse(cleaned_md, paper_id, meta_lookup)
            
            if processed_chunks:
                texts_to_embed = [c["text"] for c in processed_chunks]
                embeddings = model.encode(texts_to_embed, show_progress_bar=False)
                
                all_points = []
                for i, (chunk_data, vector) in enumerate(zip(processed_chunks, embeddings)):
                    chunk_id = hashlib.md5(f"{paper_id}_{i}_{COLLECTION_NAME}".encode()).hexdigest()
                    
                    all_points.append(PointStruct(
                        id=chunk_id,
                        vector=vector.tolist(),
                        payload={
                            "paper_id": paper_id,
                            "text_llm": chunk_data["text"],
                            "title": chunk_data["metadata"]["title"],
                            "section": chunk_data["metadata"]["section"],
                            "chunk_size": chunk_data["metadata"]["chunk_size"],
                            "chunk_method": chunk_data["metadata"]["chunk_method"]
                        }
                    ))
                
                for k in range(0, len(all_points), UPLOAD_BATCH_SIZE):
                    chunk_slice = all_points[k:k + UPLOAD_BATCH_SIZE]
                    for attempt in range(3):
                        try:
                            client.upsert(collection_name=COLLECTION_NAME, points=chunk_slice)
                            break
                        except Exception as e:
                            if attempt < 2:
                                time.sleep(5)
                            else:
                                raise e

            with open(CHECKPOINT_FILE, "a") as f:
                f.write(filename + "\n")

        except Exception as e:
            print(f"⚠️ Übersprungen wegen Fehler bei {filename}: {e}")
            continue

    print(f"🏆 INGESTION FÜR {COLLECTION_NAME} MIT 3000 PAPERS REPRODUZIERBAR BEENDET!")

if __name__ == "__main__":
    main()

🖥️ Initialisiere SentenceTransformer (BAAI/bge-m3) auf GPU...
🌐 Verbinde über Kaggle Secrets mit Qdrant...
✅ Neue Collection 'docling_hierarchical_512' (INT8, On-Disk) angelegt.
📖 Lade Metadaten für das Fusing aus: /kaggle/input/datasets/sakinaahmadi/automl-ground-truth-100/automl_ground_truth_100.json
⏮️ Bereits indiziert von den 3000 Ziel-Arbeiten: 0
🔥 Starte Pipeline für die verbleibenden 3000 Dokumente...
🏆 INGESTION FÜR docling_hierarchical_512 MIT 3000 PAPERS REPRODUZIERBAR BEENDET!


---
## 6. Zusammenfassung

### Was wurde gemacht?
- **Collection:** `docling_hierarchical_512`
- **Parser:** Docling
- **Chunking:** Hybrid (Markdown-Überschriften + rekursiv, 512 Tokens)
- **Embedding:** BGE-M3 (1024 Dimensionen)
- **Quantisierung:** INT8 (Scalar Quantization)
- **Index:** HNSW (On-Disk)
- **Distanz:** Cosine Similarity
- **Daten:** 3.000 Papers

### Nächste Schritte
1. Retrieval-Evaluierung mit dieser Collection durchführen
2. Mit anderen Collections vergleichen
3. AutoML-Optimierung der Parameter